# Analisis del comportamiento de Covagro\n\nEste notebook documenta el proyecto final: despliegue de Covagro con Docker Compose y Kubernetes, ejecucion de pruebas JMeter y analisis comparativo de tiempo medio de respuesta y throughput.

## Objetivos\n\n- Desplegar una aplicacion REST con persistencia en Docker Compose.\n- Desplegar la misma aplicacion en Kubernetes variando nodos y replicas.\n- Medir tiempo medio de respuesta, throughput, errores y percentil 95 con JMeter.\n- Comparar el efecto del escalado horizontal sobre el rendimiento.

## Aplicacion\n\nCovagro SII usa Django REST para la API, React/Vite para el frontend y PostgreSQL como base de datos en los despliegues contenedorizados. Para las pruebas se generan datos con `python manage.py seed_data`, incluyendo usuarios, productos, pedidos y movimientos de inventario.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nplt.style.use('seaborn-v0_8-whitegrid')\nresults_path = Path('../jmeter/results_template.csv')\ndf = pd.read_csv(results_path)\ndf

## Metodologia de carga\n\nEl plan JMeter `jmeter/covagro-load-test.jmx` realiza autenticacion JWT y consulta endpoints de productos, pedidos y stock bajo. Cada corrida debe registrar: escenario, nodos, replicas, usuarios concurrentes, promedio de respuesta, throughput, porcentaje de errores y percentil 95.

In [ ]:
metric_cols = ['avg_response_ms', 'throughput_req_s', 'error_pct', 'p95_response_ms']\nnumeric = df.copy()\nfor col in metric_cols:\n    numeric[col] = pd.to_numeric(numeric[col], errors='coerce')\n\nnumeric.dropna(subset=['avg_response_ms', 'throughput_req_s'], how='all')

In [ ]:
plot_df = numeric.dropna(subset=['avg_response_ms'])\nif plot_df.empty:\n    print('Completa jmeter/results_template.csv con los resultados reales de JMeter para generar graficos.')\nelse:\n    ax = plot_df.pivot_table(index='replicas', columns='scenario', values='avg_response_ms', aggfunc='mean').plot(kind='bar', figsize=(10, 5))\n    ax.set_title('Tiempo medio de respuesta por replicas')\n    ax.set_xlabel('Replicas backend')\n    ax.set_ylabel('ms')\n    plt.tight_layout()

In [ ]:
plot_df = numeric.dropna(subset=['throughput_req_s'])\nif plot_df.empty:\n    print('Completa jmeter/results_template.csv con throughput para generar graficos.')\nelse:\n    ax = plot_df.pivot_table(index='replicas', columns='scenario', values='throughput_req_s', aggfunc='mean').plot(kind='bar', figsize=(10, 5))\n    ax.set_title('Throughput por replicas')\n    ax.set_xlabel('Replicas backend')\n    ax.set_ylabel('req/s')\n    plt.tight_layout()

In [ ]:
summary = numeric.groupby(['scenario', 'nodes', 'replicas', 'users'], dropna=False)[metric_cols].mean().reset_index()\nsummary

## Analisis comparativo\n\nCompletar despues de ejecutar las pruebas:\n\n- Docker Compose: describir el comportamiento base.\n- Kubernetes 1 nodo: indicar si mas replicas mejoran throughput o solo compiten por CPU/RAM.\n- Kubernetes 2 nodos: comparar distribucion de pods y cambios de latencia.\n- Kubernetes 3 nodos: evaluar si aparece mejora adicional o sobrecosto de coordinacion.\n- Cuellos de botella: base de datos, CPU del backend, red, limites de recursos o autenticacion.

## Conclusiones\n\nRedactar conclusiones con base en los datos observados. Una conclusion fuerte debe mencionar el escenario con mejor throughput, el escenario con menor latencia, el punto donde agregar replicas deja de ayudar y las recomendaciones de despliegue para Covagro.